# Financial impact analysis — neobank NCM v3 replication

Mirror of the legacy `financial_impact_analysis.ipynb` (legacy home:
`data-science/models/underwriting/neobank/new_user/v3.0`), cell story intact;
the logic lives in `projects/neobank_ncm/analysis/` (tested offline — see
`tests/test_analysis.py`). Copy any function inline if you want to modify it.

Two model modes:
- **parity** — score with the production v3 artifacts; reproduces the legacy numbers.
- **trial** — score with an MLflow-logged trial model from this harness (our winner).

Data: three frozen `sandbox_hyong` snapshots + ONE live pull (`user_ltv.sql`,
FCT_DAILY_USER_LTV). Needs the VPN; offline, point `DAILY_PARQUET`/`LTV_PARQUET`
at cached frames.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from projects.neobank_ncm.analysis import data, impact, policy, scoring

# ── knobs ────────────────────────────────────────────────────────────────
MODEL_MODE       = 'parity'          # 'parity' | 'trial'
LEGACY_ARTIFACTS = '../../../../data-science/models/underwriting/neobank/new_user/v3.0/artifacts'
MODEL_RUN_ID     = ''                # trial mode: MLflow run id of the winner
PICKED_SCENARIO  = 2                 # 1–7, see policy.scenario_map
SAMPLING_RATE    = 1.00              # fraction of new-links volume; 1.0 = full rollout
DAILY_PARQUET    = None              # offline cache for the daily snapshot
LTV_PARQUET      = None              # offline cache for the LTV pull

pd.set_option('display.max_columns', 100)


## Load the daily snapshot (legacy cell 6)


In [ ]:
DAILY = data.load_daily(parquet=DAILY_PARQUET)

n_users = DAILY['user_id'].nunique()
n_known = DAILY.loc[DAILY['is_known'], 'user_id'].nunique()
print(f'Loaded {len(DAILY):,} rows  |  {n_users:,} users  '
      f'(known={n_known:,}  unknown={n_users - n_known:,})  columns={len(DAILY.columns)}')
print(f"Day coverage: {DAILY['day_number'].min()} – {DAILY['day_number'].max()}  "
      f"avg {DAILY.groupby('user_id')['day_number'].count().mean():.1f} days/user")


## Synthetic-score calibration on known D2 users (legacy cells 7–8)


In [ ]:
cal = scoring.calibration_table(DAILY)
d2_known = DAILY[DAILY['is_known'] & DAILY['synthetic_score'].notna() & (DAILY['day_number'] == 2)]
print(f'Known D2 users with synthetic score: {len(d2_known):,}')
print(f'Overall mean synthetic score : {d2_known["synthetic_score"].mean():.4f}')
print(f'Overall actual bad rate      : {d2_known["went_dpd45"].astype(float).mean():.4f}')
cal


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(cal['mean_syn_score'], cal['actual_br'], marker='o', linewidth=2,
        color='steelblue', label='Model (decile bins)')
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1, label='Perfect calibration')
for _, r in cal.iterrows():
    ax.annotate(f"{int(r['decile'])}", xy=(r['mean_syn_score'], r['actual_br']),
                xytext=(4, 4), textcoords='offset points', fontsize=8, color='steelblue')
ax.set_xlabel('Mean synthetic score (predicted)')
ax.set_ylabel('Actual bad rate (went DPD45)')
ax.set_title(f'Calibration curve — known D2 users (n={int(cal["n"].sum()):,})')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_xlim(0, None); ax.set_ylim(0, None)
fig.tight_layout(); plt.show()


## Score the daily snapshot (legacy cells 9–10)


In [ ]:
if MODEL_MODE == 'parity':
    model = scoring.LegacyArtifactsModel(LEGACY_ARTIFACTS)
else:
    model = scoring.TrialModel(MODEL_RUN_ID)

scoring.score_daily(DAILY, model)
print(f"Scored {len(DAILY):,} rows.  v3_score range: "
      f"[{DAILY['v3_score'].min():.4f}, {DAILY['v3_score'].max():.4f}]  "
      f"p50={DAILY['v3_score'].quantile(0.50):.4f}  p90={DAILY['v3_score'].quantile(0.90):.4f}")

auc = scoring.d2_known_auc(DAILY)
print(f"D2 AUC on known OOT users: {auc['d2_auc']:.4f}  "
      f"(n={auc['d2_n']:,}  bad_rate={auc['d2_bad_rate']:.3f})")
print('Legacy parity reference: 0.6935' if MODEL_MODE == 'parity' else f'Model: {MODEL_RUN_ID}')


## Collapse to user level + incumbent benchmarks (legacy cells 12–13)


In [ ]:
policy.add_policy_columns(DAILY)
USERS = policy.collapse_to_users(DAILY)
N = len(USERS)
masks = policy.eligibility_masks(USERS)
print(f"User-level population: {N:,}  (known={USERS['is_known'].sum():,}  "
      f"unknown={(~USERS['is_known']).sum():,})")
print(f"Avg days with data: {USERS['days_with_data'].mean():.1f}")
for label, col in [('v3a KOs', 'ko'), ('income>500', 'ko500'), ('income>500 broad', 'ko500_broad')]:
    print(f'users with ≥1 {label} passing day: {int(masks[col].sum()):,}')


In [ ]:
BENCH = policy.benchmarks(USERS)
bench_view = pd.DataFrame(
    {
        'n approved': [BENCH['n_v3a'], BENCH['n_cle']],
        'any-day AR': [BENCH['v3a_ar'], BENCH['cle_ar']],
        'D1 AR': [BENCH['v3a_d1_ar'], BENCH['cle_d1_ar']],
        'bad rate (known)': [BENCH['v3a_br'], BENCH['cle_br']],
    },
    index=['V3A (v2<=0.485 + KOs)', 'CLE (v2<=0.64 + inc>500)'],
)
bench_view.style.format({'n approved': '{:,}', 'any-day AR': '{:.1%}',
                         'D1 AR': '{:.1%}', 'bad rate (known)': '{:.1%}'})


## Threshold analysis — no-KO / income>500 (+broad) / v3a KOs (legacy cells 14–16)


In [ ]:
THRESHOLDS = policy.compute_thresholds(USERS, BENCH)
TABLES = policy.all_threshold_tables(USERS, THRESHOLDS)

fmt = {'v3_thr': '{:.4f}', 'n_v3': '{:,}', 'swap_in_vol': '{:,}', 'swap_out_vol': '{:,}'}
fmt.update({c: '{:.1%}' for c in ['v3_ar', 'ref_ar', 'v3_br', 'ref_br',
                                   'swap_in_br', 'swap_out_br', 'd1_ar']})
fmt.update({c: '{:+.1%}' for c in ['delta_ar', 'delta_br']})
for name, table in TABLES.items():
    print(f'=== {name} ===')
    display(table.style.format(fmt))


## LTV merge + historical reference (legacy cells 18–20)

`user_ltv.sql` is the one live (non-snapshot) pull: production
`FCT_DAILY_USER_LTV` joined to the frozen daily population.


In [ ]:
ltv_data = data.load_user_ltv(parquet=LTV_PARQUET)
US_RAW = impact.merge_ltv(USERS, ltv_data)
n_with_ltv = ltv_data['user_id'].isin(US_RAW['user_id']).sum()
print(f'OOT users: {len(US_RAW):,}   with LTV record: {n_with_ltv:,}   '
      f"activated: {US_RAW['is_activated'].sum():,} ({US_RAW['is_activated'].mean():.1%})")


In [ ]:
REF = impact.historical_reference(US_RAW)
print(f"Monthly new NCM neobank links: {REF['monthly_vol']:,.0f}")
print(f"Activation rate among approved (UW or CLE): {REF['act_rate']:.1%}")
pd.DataFrame(
    {
        'n elig': [REF[f'n_elig_{h}'] for h in impact.HORIZONS],
        'revenue': [REF[f'mean_rev_{h}'] for h in impact.HORIZONS],
        'loss': [REF[f'mean_loss_{h}'] for h in impact.HORIZONS],
        'LTV lite': [REF[f'mean_ltv_{h}'] for h in impact.HORIZONS],
        'LTV/link': [REF[f'lpl_{h}'] for h in impact.HORIZONS],
    },
    index=[f'D{h}' for h in impact.HORIZONS],
).style.format('{:,.2f}', subset=['revenue', 'loss', 'LTV lite', 'LTV/link'])


## Scenario pick + first-approval days (legacy cell 21)


In [ ]:
scen_label, thr_uw, thr_cle, ko_col_uw, ko_col_cle = policy.scenario_map(THRESHOLDS)[PICKED_SCENARIO]
ARRAYS = policy.first_approval_days(DAILY, USERS['user_id'], thr_uw, thr_cle, ko_col_uw, ko_col_cle)

n_uw  = int((ARRAYS['uw'] <= 30).sum())
n_cle = int((ARRAYS['cle'] <= 30).sum())
v3_ar_full  = float((ARRAYS['v3'] <= 30).sum()) / N
print(f'Scenario {PICKED_SCENARIO}: {scen_label}  |  sampling={SAMPLING_RATE:.0%}')
print(f'  Proposed UW  (${policy.LAM_UW:.0f}):  {n_uw:>7,}  {n_uw/N:.1%} AR   thr={thr_uw:.4f}')
print(f'  Proposed CLE (${policy.LAM_CLE:.0f}):  {n_cle:>7,}  {n_cle/N:.1%} AR   thr={thr_cle:.4f}')


## Lookup table + per-user inference + monthly aggregate (legacy cells 22–23)


In [ ]:
US, LKP = impact.build_lookup(US_RAW)
uw_mask  = US['user_id'].isin(set(USERS['user_id'][ARRAYS['uw'] <= 30]))
cle_only = US['user_id'].isin(set(USERS['user_id'][ARRAYS['cle'] <= 30])) & ~uw_mask
v3_mask  = uw_mask | cle_only
lam_v3 = pd.Series(np.where(uw_mask, policy.LAM_UW,
                            np.where(cle_only, policy.LAM_CLE, np.nan)), index=US.index)
V3_INF = impact.infer_financials(US, LKP, v3_mask, lam_v3)

AGG = impact.monthly_aggregate(V3_INF, N, REF['act_rate'], REF['monthly_vol'] * SAMPLING_RATE)
print(f"OOT approved: {AGG['n_app']:,} ({AGG['ar']:.1%} AR) → "
      f"{AGG['n_act']:,} projected activations (ACT_RATE={REF['act_rate']:.1%})")
pd.DataFrame(
    {
        'revenue': [AGG[f'mo_rev_{h}'] for h in impact.HORIZONS],
        'loss': [AGG[f'mo_loss_{h}'] for h in impact.HORIZONS],
        'LTV lite': [AGG[f'mo_ltv_{h}'] for h in impact.HORIZONS],
        'LTV/link': [AGG[f'lpl_{h}'] for h in impact.HORIZONS],
        'LTV/activation': [AGG[f'ltv_per_act_{h}'] for h in impact.HORIZONS],
    },
    index=[f'D{h}' for h in impact.HORIZONS],
).style.format('{:,.2f}')


## Revenue decomposition vs the V3A reference (legacy cell 24)


In [ ]:
DECOMP = impact.revenue_decomposition(REF['act_us'], V3_INF)
print('Population breakdown:')
print(f"  V3A activated (reference): {DECOMP['n_ref']:,}")
print(f"  V3 approved (proposed):    {DECOMP['n_v3']:,}")
for name, count in DECOMP['counts'].items():
    print(f'    {name:<8} {count:,}')
display(DECOMP['per_group_revenue'].style.format('{:,.2f}'))
DECOMP['decomposition'].style.format('{:+.2f}')


## Approval curve D2–D30 (legacy cell 26)


In [ ]:
curves = policy.approval_curves(ARRAYS)
fig, ax = plt.subplots(figsize=(9, 4))
styles = {'v3a': ('V3A (reference)', 'steelblue', '-', 2),
          'uw': (f'V3 UW  (${policy.LAM_UW:.0f}, thr={thr_uw:.4f})', 'green', '--', 1.5),
          'cle': (f'V3 CLE (${policy.LAM_CLE:.0f}, thr={thr_cle:.4f})', 'darkorange', '--', 1.5),
          'v3': ('V3 Combined (UW ∪ CLE)', 'darkred', '-', 2)}
for key, (label, color, ls, lw) in styles.items():
    ax.plot(curves.index, curves[key] * 100, label=label, color=color, linestyle=ls, linewidth=lw)
ax.set_xlabel('Day (D2 = first UW report date)')
ax.set_ylabel('Cumulative approval rate (%)')
ax.set_title(f'Approval curve D2–D30  |  Scenario {PICKED_SCENARIO}: {scen_label}')
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); plt.show()


## Sample size for the live experiment (legacy cell 28)


In [ ]:
lam_all = US['loan_amount_max'].fillna(US['loan_amount_max'].median())
V3A_INF = impact.infer_financials(US, LKP, US['v3a_approved'], lam_all)
SIZING = impact.sample_size_analysis(
    US, V3A_INF, V3_INF,
    p_control=float((ARRAYS['v3a'] <= 30).sum()) / N,
    p_treatment=float((ARRAYS['v3'] <= 30).sum()) / N,
    monthly_vol=REF['monthly_vol'],
)
print(f"alpha={SIZING['alpha']}  power={SIZING['power']}")
print(f"Approval rate: control {SIZING['p_control']:.1%} → treatment {SIZING['p_treatment']:.1%}")
for h, t in SIZING['ltv_tests'].items():
    print(f"LTV/link D{h}: control ${t['mu_ctrl']:.2f}±{t['ci_ctrl']:.2f}  "
          f"treatment ${t['mu_trt']:.2f}±{t['ci_trt']:.2f}  "
          f"delta ${t['mu_trt'] - t['mu_ctrl']:+.2f}±{t['ci_diff']:.2f}")
print()
for label, n in SIZING['n_per_test'].items():
    tag = '  ← binding' if label == SIZING['binding'] else ''
    print(f'  {label:<26} {n:>12,.0f}{tag}')
print(f"  total (both arms): {SIZING['n_total']:,.0f}")
SIZING['months_table'].style.format({'holdout_rate': '{:.0%}',
                                     'monthly_sampled': '{:,.0f}', 'months_needed': '{:.1f}'})
